In [1]:
from pysammos.utils.config_loader import load_config
from pysammos.coarse_graining import CoarseGraining

Hello from pysammos
Loading macroscopic_fields package...
Loading data_read package...
Loading particle_phase package...
Loading spatial_weights package...
Loading neighbour_search package...
Loading grid_generation package...
Loading data_handle package...
Loading data_write package...


In [2]:
import numpy as np

In [10]:
# Load the configuration from the ini file
cfg = load_config("./config.ini")  
print("-------------------- config.ini file read --------------------")
# Initialize the CoarseGraining class with the loaded configuration
CG = CoarseGraining(
    particle_path=cfg["particles_path"],
    contacts_path=cfg["contacts_path"],
    output_path=cfg["output_path"],
    start_timestep=cfg["t0"],
    end_timestep=cfg["tf"],
    dt_time_step=cfg["dt"],
    t_pad=cfg["t_pad"],
    DEM_keymap=cfg["key_mapping"],
    DEM_software=cfg["DEM_software"],
    grid_info=cfg["grid_info"],
    weight_function=cfg["smoothing_function"],
    fields_to_export=cfg["fields_to_export"],
    time_average=cfg["time_average"],
    ignore_phases=cfg["partialignore"],
    h5_output=cfg["h5_output"],
    vkthdf_output=cfg["vkthdf_output"],
                    ) 
print("  ") ; print("-------------------- CoarseGraining class initialised --------------------")

-------------------- config.ini file read --------------------
Output path already exists: ./PysammosCG/
  
-------------------- CoarseGraining class initialised --------------------


In [11]:
# 1. Load the size-relevant particle data for the first time step
Bounds_t0, Diameter_t0, Density_t0, Mass_t0, GlobalID_t0 = CG.data_sampling()

Sampling data for the first time step to obtain particle properties and bounds ... 
XML-based PolyData detected.


In [12]:
# 2. Calculate the particle size range
CG.get_particle_size_statistics(Diameter_t0, Mass_t0)
print(">> Particle size statistics: ") 
print("       d43: ", CG.d43)
print("       dmax: ", CG.dmax)
print("       d50: ", CG.d50)
print("       d32: ", CG.d32)
print("       drms: ", CG.drms)

>> Particle size statistics: 
       d43:  0.01651367411185649
       dmax:  0.019993
       d50:  0.016306963
       d32:  0.016109877148148044
       drms:  0.015422490500786803


In [13]:
# 3. Get the phases
CG.get_particle_phases(Diameter_t0, Density_t0, GlobalID_t0, 8)
print(">> Phases: ")
print("       Diameters: ", CG.phases[:,0])
print("       Densities: ", CG.phases[:,1])

Ignoring phases. Using mean density and d50 as phase.
>> Phases: 
       Diameters:  [0.01610988]
       Densities:  [2700.]


In [14]:
# 4. Calculate the CG grid spacing
CG.set_resolution(CG.d32) # here you can input different number, to make w and c bigger or smaller 
print(">> Coarse Graining resolution: ")
print("       c:", CG.c)
print("       w:", CG.w)

>> Coarse Graining resolution: 
       c: 0.024164815722222066
       w: 0.012082407861111033


In [15]:
# 5. Generate the CG grid
CG.generate_grid()
print(">> Grid: ")
print("       Grid Points: ", CG.GridPoints.shape, "First Point: ", CG.GridPoints[0])
print("       Nodes: ", CG.Nodes)
print("       Spacing: ", CG.Spacing)

Generating Grid with Customised Grid Ranges
Customised grid bounds: x = [0.00105, 0.5], y = [0.001, 0.24], z = [None, None], x_transect = None, y_transect = None, z_transect = 0.0
>> Grid: 
       Grid Points:  (903, 3) First Point:  [0.00105 0.001   0.     ]
       Nodes:  [43 21  1]
       Spacing:  [0.01187976 0.01195   ]


In [16]:
# 6. Calculate the CG fields
CG.fields_in_time()

 
-------------------- Calculating Coarse Grained Fields --------------------
 
---> Time step 0: time 0150 ................................................
Loading data ... 
  Particle data loaded
  Repeated pairs in contact data:  1142
  Contact data loaded and mapped
  Coordination number not required. Skipping its calculation.
... data loaded
Matching particles to grid points ...
... particles assigned to grid nodes
Computing weights ...
  Using Lucy kernel with cutoff 0.024164815722222066 and search sampling factor 1000
... weights computed
Computing Coarse Graining fields...
  volume fraction done
  mixture density done
  momentum density done
  velocity done
  velocity gradient done
  kinetic tensor done
  contact tensor done
  d43 done
  d32 done
  frabric tensor done
  particle density done
  total stress done
  pressure done
  granular temp done
  shear rate done
  inertial number done
  mu done
... fields computed
Writing results for timestep 150...
  File successfully updat

/exports/csce/datastore/geos/users/s1857688/Coarse_Graining/pysammos/pysammos/macroscopic_fields/gridded/derived_fields.py:216: RuntimeWarning: divide by zero encountered in divide
  out[f"frictional_coefficient_{s_name}_P{p_name}"] = tau / out[f"pressure_{p_name}"]


  File successfully written to ./PysammosCG/CG_Lucy_Monodisperse_0151.vtkhdf
... results written
... primary fields added to time average accumulator
  File successfully updated to ./PysammosCG/CG_GranularTemperature_slices.h5
  File successfully written to ./PysammosCG/CG_GranularTemperature_slices.h5
>> time step 1 took 0.11266756057739258 to run.
  
---> Time step 2: time 0152 ................................................
Loading data ... 
  Particle data loaded
  Repeated pairs in contact data:  1165
  Contact data loaded and mapped
  Coordination number not required. Skipping its calculation.
... data loaded
Matching particles to grid points ...
... particles assigned to grid nodes
Computing weights ...
  Using Lucy kernel with cutoff 0.024164815722222066 and search sampling factor 1000
... weights computed
Computing Coarse Graining fields...
  volume fraction done
  mixture density done
  momentum density done
  velocity done
  velocity gradient done
  kinetic tensor done
  co

## open time average results

In [18]:
# open 
from pysammos.data_write.h5.writer import H5XarrayManager 
manager = H5XarrayManager("/exports/csce/datastore/geos/users/s1857688/Coarse_Graining/pysammos/examples/bedload_transport/PysammosCG/CG_Lucy_Monodisperse_time_averaged.h5")
bedload_CG = manager.h5_to_xarray()

Processing key: density_mixture
Processing key: fabric_tensor
Processing key: frictional_coefficient_01_Pxy
Processing key: frictional_coefficient_01_Pxyz
Processing key: frictional_coefficient_01_Py
Processing key: frictional_coefficient_Dxy_Pxy
Processing key: frictional_coefficient_Dxy_Pxyz
Processing key: frictional_coefficient_Dxy_Py
Processing key: frictional_coefficient_Dxyz_Pxy
Processing key: frictional_coefficient_Dxyz_Pxyz
Processing key: frictional_coefficient_Dxyz_Py
Processing key: granular_temperature_x
Processing key: granular_temperature_y
Processing key: granular_temperature_z
Processing key: inertial_number_Sxy_Pxy_d32
Processing key: inertial_number_Sxy_Pxy_d43
Processing key: inertial_number_Sxy_Pxyz_d32
Processing key: inertial_number_Sxy_Pxyz_d43
Processing key: inertial_number_Sxy_Py_d32
Processing key: inertial_number_Sxy_Py_d43
Processing key: positions
Processing key: pressure_x
Processing key: pressure_xy
Processing key: pressure_xyz
Processing key: pressure

In [19]:
bedload_CG

<xarray.Dataset> Size: 275kB
Dimensions:                           (time: 1, point: 903, dim1_3D: 3,
                                       dim2_3D: 3, dim1_2D: 2, dim2_2D: 2,
                                       dim_1_velocity_gradient: 903,
                                       dim_2_velocity_gradient: 3,
                                       dim_3_velocity_gradient: 2, xyz: 3)
Coordinates:
  * time                              (time) int64 8B 150
  * point                             (point) int64 7kB 0 1 2 3 ... 900 901 902
    positions                         (point, xyz) float64 22kB 0.00105 ... 0.0
Dimensions without coordinates: dim1_3D, dim2_3D, dim1_2D, dim2_2D,
                                dim_1_velocity_gradient,
                                dim_2_velocity_gradient,
                                dim_3_velocity_gradient, xyz
Data variables: (12/34)
    density_mixture                   (time, point) float32 4kB 420.7 ... 0.0
    fabric_tensor                     (time, point, dim1_3D, dim2_3D) float32 33kB ...
    frictional_coefficient_01_Pxy     (time, point) float32 4kB 0.04003 ... nan
    frictional_coefficient_01_Pxyz    (time, point) float32 4kB 0.06004 ... nan
    frictional_coefficient_01_Py      (time, point) float32 4kB 0.03657 ... nan
    frictional_coefficient_Dxy_Pxy    (time, point) float32 4kB 0.1028 ... nan
    ...                                ...
    total_stress_tensor_xyz_dev       (time, point, dim1_3D, dim2_3D) float32 33kB ...
    total_stress_tensor_xyz_dev_mag   (time, point) float32 4kB 49.91 ... 0.0
    total_stress_tensor_xyz_mag       (time, point) float32 4kB 85.56 ... 0.0
    velocity                          (time, point, dim1_3D) float32 11kB 0.0...
    velocity_gradient                 (time, dim_1_velocity_gradient, dim_2_velocity_gradient, dim_3_velocity_gradient) float32 22kB ...
    volume_fraction                   (time, point) float32 4kB 0.1558 ... 0.0